# Interactive Brokers — Historical Data Pull

Connect to IB via DockerizedIBGateway and pull historical bar data.
Saves to CSV format compatible with NautilusTrader's `BarDataWrangler`.

## Prerequisites
- Docker Desktop running
- `.env.ib` file with `TWS_USERNAME` and `TWS_PASSWORD`
- IB account with market data subscriptions

In [ ]:
import asyncio
import datetime
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from nautilus_trader.adapters.interactive_brokers.common import IBContract
from nautilus_trader.adapters.interactive_brokers.config import DockerizedIBGatewayConfig
from nautilus_trader.adapters.interactive_brokers.gateway import DockerizedIBGateway
from nautilus_trader.adapters.interactive_brokers.historical.client import (
    HistoricInteractiveBrokersClient,
)

# Load IB credentials from .env.ib
load_dotenv(dotenv_path=Path(__file__).parent.parent / ".env.ib" if "__file__" in dir() else ".env.ib")

DATA_DIR = Path("../../../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"TWS_USERNAME: {'set' if os.getenv('TWS_USERNAME') else 'NOT SET'}")
print(f"TWS_PASSWORD: {'set' if os.getenv('TWS_PASSWORD') else 'NOT SET'}")
print(f"Data directory: {DATA_DIR.resolve()}")

## 1. Start IB Gateway (Docker)

This starts a containerized IB Gateway. It uses your `.env.ib` credentials.
The container exposes port 4001 (live) for API connections.

> **Note:** First run will pull the Docker image (~1.5 GB). Subsequent starts are fast.
> The gateway takes ~30-60 seconds to fully authenticate.

In [ ]:
gateway_config = DockerizedIBGatewayConfig(
    username=os.getenv("TWS_USERNAME"),
    password=os.getenv("TWS_PASSWORD"),
    trading_mode="live",
    read_only_api=True,
    timeout=300,
)

gateway = DockerizedIBGateway(config=gateway_config)
gateway.safe_start()
print(f"Gateway running on {gateway.host}:{gateway.port}")

## 2. Connect Historical Data Client

Connects to the running IB Gateway for historical data requests.

In [ ]:
client = HistoricInteractiveBrokersClient(
    host=gateway.host,
    port=gateway.port,
    client_id=1,
)

await client.connect()
print("Connected to IB Gateway")

## 3. Pull Historical Data

Configure the instrument, bar size, and date range below.
Two examples are provided: XAUUSD 1-minute and 5-minute bars.

### IB Rate Limits
- Max 60 requests per 10 minutes
- No identical requests within 15 seconds
- The client handles chunked requests automatically

### Available Bar Specifications
- `1-MINUTE-BID`, `1-MINUTE-LAST`, `1-MINUTE-MID`
- `5-MINUTE-BID`, `5-MINUTE-LAST`, `5-MINUTE-MID`
- `1-HOUR-BID`, `1-HOUR-LAST`, `1-HOUR-MID`
- `1-DAY-BID`, `1-DAY-LAST`, `1-DAY-MID`

In [ ]:
# XAUUSD (Spot Gold) — available for non-US residents
# For US residents, use: IBContract(secType="FUT", symbol="GC", exchange="COMEX", lastTradeDateOrContractMonth="YYYYMM")
xauusd_contract = IBContract(
    secType="CFD",
    symbol="XAUUSD",
    exchange="SMART",
    currency="USD",
)

# Date range — adjust as needed
# IB provides up to ~5 years of 1-minute data
end_date = datetime.datetime.now()
start_date = end_date - datetime.timedelta(days=365)  # 1 year

print(f"Contract: {xauusd_contract.symbol}")
print(f"Date range: {start_date.date()} to {end_date.date()}")

### Example 1: Pull 1-Minute Bars

In [ ]:
bars_1m = await client.request_bars(
    bar_specifications=["1-MINUTE-BID"],
    start_date_time=start_date,
    end_date_time=end_date,
    tz_name="UTC",
    contracts=[xauusd_contract],
    use_rth=False,  # Include extended hours for forex/CFDs
    timeout=120,
)

print(f"Received {len(bars_1m)} 1-minute bars")
if bars_1m:
    print(f"First bar: {bars_1m[0]}")
    print(f"Last bar:  {bars_1m[-1]}")

In [ ]:
if bars_1m:
    df_1m = pd.DataFrame([
        {
            "timestamp": pd.Timestamp(bar.ts_event, unit="ns", tz="UTC"),
            "open": float(bar.open),
            "high": float(bar.high),
            "low": float(bar.low),
            "close": float(bar.close),
            "volume": float(bar.volume),
        }
        for bar in bars_1m
    ])

    filepath_1m = DATA_DIR / "xauusd_1m.csv"
    df_1m.to_csv(filepath_1m, index=False)
    print(f"Saved {len(df_1m)} bars to {filepath_1m.resolve()}")
    df_1m.head()

### Example 2: Pull 5-Minute Bars

In [ ]:
bars_5m = await client.request_bars(
    bar_specifications=["5-MINUTE-BID"],
    start_date_time=start_date,
    end_date_time=end_date,
    tz_name="UTC",
    contracts=[xauusd_contract],
    use_rth=False,
    timeout=120,
)

print(f"Received {len(bars_5m)} 5-minute bars")
if bars_5m:
    print(f"First bar: {bars_5m[0]}")
    print(f"Last bar:  {bars_5m[-1]}")

In [ ]:
if bars_5m:
    df_5m = pd.DataFrame([
        {
            "timestamp": pd.Timestamp(bar.ts_event, unit="ns", tz="UTC"),
            "open": float(bar.open),
            "high": float(bar.high),
            "low": float(bar.low),
            "close": float(bar.close),
            "volume": float(bar.volume),
        }
        for bar in bars_5m
    ])

    filepath_5m = DATA_DIR / "xauusd_5m.csv"
    df_5m.to_csv(filepath_5m, index=False)
    print(f"Saved {len(df_5m)} bars to {filepath_5m.resolve()}")
    df_5m.head()

### Custom Pull

Modify the contract, bar spec, and date range below to pull any instrument.

In [ ]:
# === CONFIGURE YOUR PULL HERE ===
custom_contract = IBContract(
    secType="CFD",       # STK, FUT, OPT, CFD, CASH
    symbol="XAUUSD",     # IB symbol
    exchange="SMART",    # Exchange
    currency="USD",      # Currency
)

custom_bar_specs = ["1-MINUTE-BID"]  # Change timeframe here
custom_start = datetime.datetime.now() - datetime.timedelta(days=30)  # 30 days
custom_end = datetime.datetime.now()
custom_filename = "custom_data.csv"
# ================================

custom_bars = await client.request_bars(
    bar_specifications=custom_bar_specs,
    start_date_time=custom_start,
    end_date_time=custom_end,
    tz_name="UTC",
    contracts=[custom_contract],
    use_rth=False,
    timeout=120,
)

print(f"Received {len(custom_bars)} bars")

if custom_bars:
    df_custom = pd.DataFrame([
        {
            "timestamp": pd.Timestamp(bar.ts_event, unit="ns", tz="UTC"),
            "open": float(bar.open),
            "high": float(bar.high),
            "low": float(bar.low),
            "close": float(bar.close),
            "volume": float(bar.volume),
        }
        for bar in custom_bars
    ])

    filepath_custom = DATA_DIR / custom_filename
    df_custom.to_csv(filepath_custom, index=False)
    print(f"Saved {len(df_custom)} bars to {filepath_custom.resolve()}")
    df_custom.head()

## 4. Cleanup

Stop the IB Gateway Docker container when done.

In [ ]:
gateway.safe_stop()
print("IB Gateway stopped")